# Step 1

In [ ]:
import kagglehub
path = kagglehub.dataset_download("yasserh/housing-prices-dataset")

print(path)

Using Colab cache for faster access to the 'housing-prices-dataset' dataset.
/kaggle/input/housing-prices-dataset


In [ ]:
import numpy as np
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
df = pd.read_csv("/kaggle/input/housing-prices-dataset/Housing.csv")
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


# Step 2

In [ ]:
fig = px.scatter(df, x="area", y="price", title="House Price vs Area",
    labels={
        "Area": "Area",
        "Price": "Price"
    },
    hover_data=["bedrooms", "bathrooms", "stories"]
)

fig.show()

In [ ]:
fig = px.box(
    df,
    x="stories",
    y="price",
    title="House Price by Number of Floors",
    labels={
        "Floors": "Number of Floors",
        "Price": "Price"
    }
)

fig.show()

In [ ]:
ndf = df.select_dtypes(include="number")
nc = ndf.corr()
nc

,price,area,bedrooms,bathrooms,stories,parking
price,1.000000,0.535997,0.366494,0.517545,0.420712,0.384394
area,0.535997,1.000000,0.151858,0.193820,0.083996,0.352980
bedrooms,0.366494,0.151858,1.000000,0.373930,0.408564,0.139270
bathrooms,0.517545,0.193820,0.373930,1.000000,0.326165,0.177496
stories,0.420712,0.083996,0.408564,0.326165,1.000000,0.045547
parking,0.384394,0.352980,0.139270,0.177496,0.045547,1.000000


In [ ]:
fig = px.imshow(
    nc,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap of Numeric Features",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)

fig.show()

# Step 3

In [ ]:
df.isna().sum()

,0
price,0
area,0
bedrooms,0
bathrooms,0
stories,0
mainroad,0
guestroom,0
basement,0
hotwaterheating,0
airconditioning,0


In [ ]:
df.notna().sum()

,0
price,545
area,545
bedrooms,545
bathrooms,545
stories,545
mainroad,545
guestroom,545
basement,545
hotwaterheating,545
airconditioning,545


# Step 4

In [ ]:
ndf

,price,area,bedrooms,bathrooms,stories,parking
0,13300000,7420,4,2,3,2
1,12250000,8960,4,4,4,3
2,12250000,9960,3,2,2,2
3,12215000,7500,4,2,2,3
4,11410000,7420,4,1,2,2
...,...,...,...,...,...,...
540,1820000,3000,2,1,1,2
541,1767150,2400,3,1,1,0
542,1750000,3620,2,1,1,0
543,1750000,2910,3,1,1,0


In [ ]:
sdf = df.select_dtypes(include=['object'])
sdf

,mainroad,guestroom,basement,hotwaterheating,airconditioning,prefarea,furnishingstatus
0,yes,no,no,no,yes,yes,furnished
1,yes,no,no,no,yes,no,furnished
2,yes,no,yes,no,no,yes,semi-furnished
3,yes,no,yes,no,yes,yes,furnished
4,yes,yes,yes,no,yes,no,furnished
...,...,...,...,...,...,...,...
540,yes,no,yes,no,no,no,unfurnished
541,no,no,no,no,no,no,semi-furnished
542,yes,no,no,no,no,no,unfurnished
543,no,no,no,no,no,no,furnished


# Step 5

In [ ]:
dfenc = df.copy()
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

for col in binary_cols:
    dfenc[col] = dfenc[col].map({'yes': 1, 'no': 0})

furnishing_status_mapping = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
dfenc['furnishingstatus'] = dfenc['furnishingstatus'].map(furnishing_status_mapping)

dfenc

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,2
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,2
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,1,0,1,0,0,2,0,0
541,1767150,2400,3,1,1,0,0,0,0,0,0,0,1
542,1750000,3620,2,1,1,1,0,0,0,0,0,0,0
543,1750000,2910,3,1,1,0,0,0,0,0,0,0,2


In [ ]:
dfenc_corr = dfenc.corr()

fig = px.imshow(
    dfenc_corr,
    text_auto=".2f",
    aspect="auto",
    title="Correlation Heatmap of Numeric Features",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1
)

fig.show()

# Step 6

In [ ]:
x = dfenc.drop('price', axis=1)
y = dfenc['price']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

## Elastic Net

In [ ]:
elastic_net_model = ElasticNet(random_state=42)
elastic_net_model.fit(x_train, y_train)

y_pred_en = elastic_net_model.predict(x_test)

mae_en = mean_absolute_error(y_test, y_pred_en)
mse_en = mean_squared_error(y_test, y_pred_en)
r2_en = r2_score(y_test, y_pred_en)

print("Elastic Net Regression:")
print("MAE:", mae_en)
print("MSE:", mse_en)
print("R^2:", r2_en)

Elastic Net Regression:
MAE: 1121771.7519758446
MSE: 2321416682314.5176
R^2: 0.540729168026424


## KNN

In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=6)
knn_model.fit(x_train, y_train)

y_pred_knn = knn_model.predict(x_test)

mae_knn = mean_absolute_error(y_test, y_pred_knn)
mse_knn = mean_squared_error(y_test, y_pred_knn)
r2_knn = r2_score(y_test, y_pred_knn)

print("KNN:")
print("MAE:", mae_knn)
print("MSE:", mse_knn)
print("R^2:", r2_knn)

KNN:
MAE: 1313554.2813455656
MSE: 3230178240379.7144
R^2: 0.3609390941384054
